In [1]:
%load_ext autoreload
%autoreload 2

import copy
import random
import re
from collections import Counter
from pathlib import Path
import json
from time import time, sleep
 
import mne
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

import seeg_fingerprint as sf

In [14]:
base = Path('/home/gbz6qn/Documents/PhD/Fall26/DeepLearning/dashboard/example_data/sub-HUP060')
stem = 'sub-HUP060_ses-presurgery_task-interictal_acq-seeg_run-{:02d}'

edfs = [base / (stem.format(r) + '_ieeg.edf') for r in (1, 2)]
tsv  = base / (stem.format(2) + '_channels.tsv')

channels = sf.good_channels(tsv, edfs)
print(len(channels), 'contacts')

36 contacts


In [2]:
FS = 256
WIN_SEC = 1.0
NORM = 'global'
config = dict(norm=NORM, fs=FS, win_sec=WIN_SEC, avg_ref=True)
subs = ['sub-HUP060', 'sub-HUP117', 'sub-HUP133']
first_start = time()
for sub in subs:
    start = time()
    print(f'***** {sub.upper()} *****')
    pdir = Path(f'/home/gbz6qn/Documents/PhD/Fall26/DeepLearning/dashboard/results/{sub}')
    base = Path(f'/home/gbz6qn/Documents/PhD/Fall26/DeepLearning/dashboard/example_data/{sub}')
    stem = f'{sub}_ses-presurgery_task-interictal_acq-seeg_run-{{:02d}}'

    edfs = [base / (stem.format(r) + '_ieeg.edf') for r in (1, 2)]
    tsv = base / (stem.format(2) + '_channels.tsv')

    channels = sf.good_channels(tsv, edfs)
    print(f'{sub}: {len(channels)} contacts')
    loaders, win = sf.build_loaders(edfs[0], edfs[1], channels, fs=FS, win_sec=WIN_SEC, norm=NORM)
    sf.export_patient(pdir, channels, loaders)

    x, y = next(iter(loaders['train']))
    print(x.shape, y.shape, win, {k: len(v.dataset) for k, v in loaders.items()})

    models = {
        'MLP': lambda: sf.build_mlp(win, len(channels)),
        'CNN': lambda: sf.build_simple_cnn(win, len(channels)),
        # 'Net1D': lambda: sf.build_net1d(len(channels)),
    }
    for name, make in models.items():
        print(f'Training {name}')
        sf.set_seed(0)        
        model, hist = sf.train_model(make(), loaders)
        print(name, 'val acc/loss:', sf.evaluate(model, loaders['val']))
        sf.save_results(pdir / 'results.json', name, model, hist, loaders, len(channels), config)
        sf.save_model(pdir / 'models' / f'{name}.pt', model)

    print(f'Done {sub}. Took {(time() - start) / 60:.1f} minutes')

print(f'Done all subs and models. Took {(time() - first_start) / 60:1f} minutes.')

***** SUB-HUP060 *****
sub-HUP060: 36 contacts
torch.Size([128, 1, 256]) torch.Size([128]) 256 {'train': 63468, 'val': 1980, 'test': 1980, 'run2': 10800}
Training MLP
epoch   0  train_loss 2.6943  val_loss 2.2828  val_acc 0.2323
epoch   1  train_loss 2.0616  val_loss 2.1629  val_acc 0.2773
epoch   2  train_loss 1.8968  val_loss 2.0816  val_acc 0.3005
epoch   3  train_loss 1.7830  val_loss 2.0460  val_acc 0.3167
epoch   4  train_loss 1.7039  val_loss 2.0668  val_acc 0.3025
epoch   5  train_loss 1.6301  val_loss 2.0136  val_acc 0.3172
epoch   6  train_loss 1.5577  val_loss 2.0718  val_acc 0.3182
epoch   7  train_loss 1.4928  val_loss 2.1311  val_acc 0.3111
epoch   8  train_loss 1.4316  val_loss 2.1234  val_acc 0.3343
epoch   9  train_loss 1.3706  val_loss 2.1280  val_acc 0.3202
epoch  10  train_loss 1.3190  val_loss 2.2013  val_acc 0.3293
epoch  11  train_loss 1.2694  val_loss 2.2236  val_acc 0.3192
epoch  12  train_loss 1.2148  val_loss 2.2908  val_acc 0.3278
epoch  13  train_loss 1.170

In [15]:
FS = 256
WIN_SEC = 1.0
NORM = 'global'
config = dict(norm=NORM, fs=FS, win_sec=WIN_SEC, avg_ref=True)
sf.set_seed(0)
loaders, win = sf.build_loaders(edfs[0], edfs[1], channels, fs=FS, win_sec=WIN_SEC, norm=NORM)
x, y = next(iter(loaders['train']))
print(x.shape, y.shape, win)
print({k: len(v.dataset) for k, v in loaders.items()})

torch.Size([128, 1, 256]) torch.Size([128]) 256
{'train': 63468, 'val': 1980, 'test': 1980, 'run2': 10800}


In [42]:
# Train MLP
mlp = sf.build_mlp(win, len(channels))
mlp, mlp_hist = sf.train_model(mlp, loaders)

epoch   0  train_loss 2.7032  val_loss 2.3093  val_acc 0.2192
epoch   1  train_loss 2.0653  val_loss 2.1632  val_acc 0.2747
epoch   2  train_loss 1.9002  val_loss 2.0889  val_acc 0.2869
epoch   3  train_loss 1.7828  val_loss 2.0464  val_acc 0.3121
epoch   4  train_loss 1.6965  val_loss 2.0585  val_acc 0.3051
epoch   5  train_loss 1.6190  val_loss 2.0279  val_acc 0.3146
epoch   6  train_loss 1.5442  val_loss 2.0441  val_acc 0.3162
epoch   7  train_loss 1.4773  val_loss 2.0748  val_acc 0.3419
epoch   8  train_loss 1.4183  val_loss 2.1154  val_acc 0.3283
epoch   9  train_loss 1.3571  val_loss 2.1060  val_acc 0.3232
epoch  10  train_loss 1.3002  val_loss 2.2160  val_acc 0.2990
epoch  11  train_loss 1.2464  val_loss 2.2291  val_acc 0.3212
epoch  12  train_loss 1.1966  val_loss 2.2205  val_acc 0.3242
epoch  13  train_loss 1.1464  val_loss 2.3616  val_acc 0.3253
epoch  14  train_loss 1.1015  val_loss 2.3808  val_acc 0.3313
epoch  15  train_loss 1.0576  val_loss 2.4874  val_acc 0.3227
epoch  1

In [43]:
# Train simple CNN
cnn = sf.build_simple_cnn(win, len(channels))
cnn, cnn_hist = sf.train_model(cnn, loaders)

epoch   0  train_loss 2.1305  val_loss 1.6381  val_acc 0.4040
epoch   1  train_loss 1.5449  val_loss 1.4023  val_acc 0.4753
epoch   2  train_loss 1.4013  val_loss 1.2927  val_acc 0.5076
epoch   3  train_loss 1.3377  val_loss 1.2925  val_acc 0.5035
epoch   4  train_loss 1.2963  val_loss 1.2612  val_acc 0.5187
epoch   5  train_loss 1.2704  val_loss 1.2485  val_acc 0.5192
epoch   6  train_loss 1.2437  val_loss 1.2365  val_acc 0.5303
epoch   7  train_loss 1.2204  val_loss 1.1863  val_acc 0.5348
epoch   8  train_loss 1.2039  val_loss 1.2424  val_acc 0.5263
epoch   9  train_loss 1.1941  val_loss 1.2152  val_acc 0.5354
epoch  10  train_loss 1.1746  val_loss 1.1842  val_acc 0.5485
epoch  11  train_loss 1.1710  val_loss 1.1886  val_acc 0.5490
epoch  12  train_loss 1.1510  val_loss 1.2245  val_acc 0.5515
epoch  13  train_loss 1.1434  val_loss 1.1471  val_acc 0.5606
epoch  14  train_loss 1.1340  val_loss 1.2311  val_acc 0.5288
epoch  15  train_loss 1.1308  val_loss 1.1305  val_acc 0.5626
epoch  1

In [45]:
# Train Net1D
net = sf.build_net1d(len(channels))
net, net_hist = sf.train_model(net, loaders)

epoch   0  train_loss 1.7233  val_loss 1.2888  val_acc 0.5066
epoch   1  train_loss 1.2703  val_loss 1.2680  val_acc 0.5328
epoch   2  train_loss 1.1632  val_loss 1.1742  val_acc 0.5636
epoch   3  train_loss 1.0902  val_loss 1.1395  val_acc 0.5657
epoch   4  train_loss 1.0439  val_loss 1.1055  val_acc 0.5914
epoch   5  train_loss 1.0031  val_loss 1.1001  val_acc 0.5889
epoch   6  train_loss 0.9564  val_loss 1.1054  val_acc 0.5848
epoch   7  train_loss 0.9229  val_loss 1.1351  val_acc 0.5848
epoch   8  train_loss 0.8829  val_loss 1.1417  val_acc 0.5818
epoch   9  train_loss 0.8544  val_loss 1.1025  val_acc 0.5894
epoch  10  train_loss 0.8248  val_loss 1.2093  val_acc 0.5808
epoch  11  train_loss 0.8037  val_loss 1.2402  val_acc 0.5677
epoch  12  train_loss 0.7740  val_loss 1.1893  val_acc 0.5818
epoch  13  train_loss 0.7491  val_loss 1.1845  val_acc 0.5889
epoch  14  train_loss 0.7247  val_loss 1.2520  val_acc 0.5707
Early stopping at epoch 14, best val_acc: 0.591


In [46]:
# Check and save
for name, m in [('MLP', mlp), ('SimpleCNN', cnn), ('Net1D', net)]:
    print(name, 'val acc/loss:', sf.evaluate(m, loaders['val']))

sf.save_results('results.json', 'MLP',   mlp, mlp_hist, loaders, len(channels), config)
sf.save_results('results.json', 'SimpleCNN', cnn, cnn_hist, loaders, len(channels), config)
#sf.save_results('results.json', 'Net1D', net, net_hist, loaders, len(channels), config)

MLP val acc/loss: (0.3419191919191919, 2.0748250903505268)
SimpleCNN val acc/loss: (0.5818181818181818, 1.1356904766776346)
Net1D val acc/loss: (0.5914141414141414, 1.105523365675801)
